In [ ]:
import arcpy
import os
import random

# 设置工作空间
arcpy.env.overwriteOutput = True
scratch_gdb = arcpy.env.scratchGDB

# 输入数据路径
sinkhole_points = r"D:\path\to\sinkhole-risk-china\data\labels_Geological_hazard_in_China\selected_sinkhole_merge\Selected_Sinkhole_Merge_China.shp"
china_boundary = r"D:\path\to\sinkhole-risk-china\data\Administrative_divisions_of_china\china_pro2.shp"

# 步骤1：获取中国边界范围和空间参考
print("正在获取中国边界范围和空间参考...")
sr = arcpy.Describe(china_boundary).spatialReference
china_extent = arcpy.Describe(china_boundary).extent

# 设置输出坐标系
arcpy.env.outputCoordinateSystem = sr

# 步骤2：创建多边形渔网（单元大小10km）
print("正在创建渔网...")
fishnet = os.path.join(scratch_gdb, "fishnet")
cell_size = 10000  # 10km

# 创建多边形渔网
arcpy.management.CreateFishnet(
    out_feature_class=fishnet,
    origin_coord=f"{china_extent.XMin} {china_extent.YMin}",
    y_axis_coord=f"{china_extent.XMin} {china_extent.YMax}",
    cell_width=cell_size,
    cell_height=cell_size,
    corner_coord=f"{china_extent.XMax} {china_extent.YMax}",
    geometry_type="POLYGON"
)

# 步骤3：生成渔网中心点
print("正在生成候选点...")
candidate_points = os.path.join(scratch_gdb, "candidate_points")
arcpy.management.FeatureToPoint(fishnet, candidate_points, "CENTROID")  # 使用质心而不是内部点

# 步骤4：裁剪到中国边界
print("正在裁剪点...")
clipped_points = os.path.join(scratch_gdb, "clipped_points")
arcpy.analysis.Clip(candidate_points, china_boundary, clipped_points)

# 步骤5：创建正样本10km缓冲区
print("创建缓冲区...")
sinkhole_buffer = os.path.join(scratch_gdb, "sinkhole_buffer")
arcpy.analysis.Buffer(sinkhole_points, sinkhole_buffer, "10 Kilometers")

# 步骤6：筛选有效候选点
print("筛选候选点...")
valid_points = os.path.join(scratch_gdb, "valid_points")
arcpy.analysis.Erase(clipped_points, sinkhole_buffer, valid_points)

# 步骤7：随机选择与正样本相同数量的点
print("随机选择负样本...")
num_samples = int(arcpy.management.GetCount(sinkhole_points)[0])
all_points = [row[0] for row in arcpy.da.SearchCursor(valid_points, ["OID@"])]

# 确保有足够的候选点
if len(all_points) < num_samples:
    print(f"警告：有效候选点数量({len(all_points)})少于正样本数量({num_samples})")
    print("可能需要增大渔网单元尺寸或减小缓冲区距离")
    selected_ids = all_points
else:
    selected_ids = random.sample(all_points, num_samples)

# 创建最终负样本
negative_samples = os.path.join(scratch_gdb, "negative_samples")
arcpy.management.MakeFeatureLayer(valid_points, "temp_layer")
arcpy.management.SelectLayerByAttribute("temp_layer", "NEW_SELECTION", f"OBJECTID IN ({','.join(map(str, selected_ids))})")
arcpy.management.CopyFeatures("temp_layer", negative_samples)

# 步骤8：添加到当前地图
print("添加结果到地图...")
project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap
active_map.addDataFromPath(negative_samples)

print(f"成功生成 {len(selected_ids)} 个负样本点！")

In [ ]:
# 新增步骤：添加并计算经纬度字段（使用投影和游标）
print("添加经纬度字段...")
wgs84 = arcpy.SpatialReference(4326)  # WGS84坐标系

# 将负样本点投影到WGS84坐标系
projected_negative_samples = os.path.join(scratch_gdb, "projected_negative_samples")
arcpy.management.Project(negative_samples, projected_negative_samples, wgs84)

# 添加经纬度字段
arcpy.management.AddField(projected_negative_samples, "Longitude", "DOUBLE")
arcpy.management.AddField(projected_negative_samples, "Latitude", "DOUBLE")

# 使用游标计算经纬度
with arcpy.da.UpdateCursor(projected_negative_samples, ["SHAPE@X", "SHAPE@Y", "Longitude", "Latitude"]) as cursor:
    for row in cursor:
        row[2] = row[0]  # X -> Longitude
        row[3] = row[1]  # Y -> Latitude
        cursor.updateRow(row)

# 更新负样本为投影后的点
negative_samples = projected_negative_samples

# 步骤8：添加到当前地图
print("添加结果到地图...")
project = arcpy.mp.ArcGISProject("CURRENT")
active_map = project.activeMap
active_map.addDataFromPath(negative_samples)

print(f"成功生成 {len(selected_ids)} 个负样本点，并添加了经纬度信息！")